In [1]:
%cd ../../..

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Load data

## Raw biowaste

### From 2023-01-02 -> 2024-06-11

In [3]:
path = "data/raw/biowaste/dim_biowaste.xlsx"

biowaste_day_raw = pd.read_excel(path, index_col=None)
biowaste_day_raw.head()

,date,restaurant,amnt_waste_customer,amnt_waste_coffee,amnt_waste_kitchen,amnt_waste_hall
0,2023-01-02,Chemicum,4.70,1.2,12.0,0.0
1,2023-01-03,Chemicum,5.00,1.4,14.8,0.0
2,2023-01-04,Chemicum,4.15,4.0,7.1,0.0
3,2023-01-05,Chemicum,10.00,3.3,8.5,0.0
4,2023-01-09,Chemicum,7.65,2.1,4.9,0.0


In [4]:
biowaste_day = biowaste_day_raw.copy()


# Convert restaurant
biowaste_day['restaurant'] = biowaste_day['restaurant'].map({
    'Chemicum': 'che', #'chemicum',
    'Physicum': 'phy', #'physicum',
    'Exactum': 'exa', #'exactum'
    'viikuna': 'vik',
})


# Sum waste
biowaste_day['waste'] = (
    biowaste_day['amnt_waste_customer']
    + biowaste_day['amnt_waste_coffee']
    + biowaste_day['amnt_waste_kitchen']
    + biowaste_day['amnt_waste_hall']
)


# Remove unnecessary columns
biowaste_day.drop(columns=['amnt_waste_customer', 'amnt_waste_coffee', 'amnt_waste_kitchen', 'amnt_waste_hall'], inplace=True)


biowaste_day.head()

,date,restaurant,waste
0,2023-01-02,che,17.90
1,2023-01-03,che,21.20
2,2023-01-04,che,15.25
3,2023-01-05,che,21.80
4,2023-01-09,che,14.65


### From 2024-09-02 -> 2024-10-30 (`Data.xlsx`)

In [5]:
path = "data/raw/biowaste/Data.xlsx"

waste_2425_raw = pd.read_excel(path)
waste_2425_raw.head()

,Myydyt tuotteet kpl,Date,Tuotteen id + nimi,Hävikki,Ravintola
0,1,2024-09-02,"1516 Panini, hot chili kana",10,610 Physicum
1,1,2024-09-02,1802 Buffet,54,600 Chemicum
2,1,2024-09-02,20021 Grillattu kana ohutleipärulla,10,610 Physicum
3,1,2024-09-02,3215 Take away ruoka,19,620 Exactum
4,1,2024-09-03,"10090 Liha, take away",16,620 Exactum


In [6]:
waste_2425 = waste_2425_raw.copy()


# Rename column
waste_2425.columns = ['pcs', 'date', 'meal_raw', 'waste', 'restaurant']


# Convert to datetime
waste_2425['date'] = pd.to_datetime(waste_2425['date'])

# Extract meal name and id
pat = r"(\d*)\s(.*)"
def _extract(s: str):
    out = re.findall(pat, s)

    assert len(out) > 0
    out = out[0]

    return pd.Series({'meal_id': int(out[0]), 'meal': out[1]})
out = waste_2425['meal_raw'].apply(_extract)
waste_2425 = pd.concat([waste_2425, out], axis=1).drop(columns='meal_raw')


# Convert restaurant
waste_2425['restaurant'] = waste_2425['restaurant'].map({
    '610 Physicum': 'phy',
    '600 Chemicum': 'che',
    '620 Exactum': 'exa',
})


# Get first record per date and restaurant
waste_2425 = waste_2425.groupby(['date', 'restaurant'])['waste'].first().reset_index()


waste_2425.head()

,date,restaurant,waste
0,2024-09-02,che,54
1,2024-09-02,exa,19
2,2024-09-02,phy,10
3,2024-09-03,che,44
4,2024-09-03,exa,16


### Concat historical waste of schoolyear 23-24 and 24-25

In [7]:
biowaste_day = pd.concat([biowaste_day, waste_2425], axis=0)
biowaste_day.head()

,date,restaurant,waste
0,2023-01-02,che,17.90
1,2023-01-03,che,21.20
2,2023-01-04,che,15.25
3,2023-01-05,che,21.80
4,2023-01-09,che,14.65


## dim `meals_name`

In [8]:
path = "data/processed/phase_4/dim_meal_names.xlsx"

dim_meal_names = pd.read_excel(path)
dim_meal_names.head()

,meal_id,meal
0,9017,"""Butter"" härkäpapua & pähkinää"
1,7201,2023 Härkäpu-sienilasagnette
2,9032,Appelisiini-luomukikhernecurrya
3,9102,Artisokkavugetteja & tuoretomaattisalsaa
4,7010,Aurajuusto-pinaattilasagnette


# Raw POS

In [9]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
]

raw = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = np.arange(df.shape[1])
    raw.append(df)


pos_raw = pd.concat(raw, ignore_index=True)
pos_raw.head()

/tmp/ipykernel_42916/1943815052.py:9: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,0,1,2,3,4,5,6
0,2.1.2023,10:31,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
1,2.1.2023,10:32,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
2,2.1.2023,10:32,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
3,2.1.2023,10:35,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
4,2.1.2023,10:36,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",2,"1,8"


In [10]:
pos = pos_raw.copy()


# Rename columns
pos.columns = np.arange(pos.shape[1])
pos.rename(
    columns={
        0: 'date',
        1: 'time',
        2: 'restaurant',
        3: 'meal_type',
        4: 'meal',
        5: 'pcs',
        6: 'co2',
    },
    inplace=True
)


# Convert pcs
def _f_conv(s):
    match s:
        case int() | float():
            return float(s)
        case str():
            if re.search(r'\s', s) is None:
                return float(s)
            return np.nan

pos['pcs'] = pos['pcs'].apply(_f_conv)
pos = pos[~pos['pcs'].isna()]


# Map restaurant name
pos['restaurant'] = pos['restaurant'].map({
    '600 Chemicum': 'che', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'exa', #'exactum'
    '570 Viikuna': 'vik',
})


# Process date
pos['date'] = pd.to_datetime(pos['date'], format="%d.%m.%Y")


# Process meal
pos['meal'] = pos['meal'].str.strip()


# Remove 'take away' meals
pos = pos[~pos['meal'].str.lower().str.contains('take away')]



# Accumulate pcs per day
pos = (
    pos
    .groupby(['date', 'restaurant', 'meal'])['pcs']
    .sum()
    .reset_index()
)


# Get meal_id and remove records having no meal_id
pos = (
    pos
    .merge(dim_meal_names, on='meal', how='left')
    .dropna(axis=0, how='any')
)



# Keep necessary columns
pos.drop(columns=['meal'], inplace=True)


pos.head()

,date,restaurant,pcs,meal_id
0,2023-01-02,che,78.0,9500047.0
2,2023-01-02,che,84.0,6128.0
3,2023-01-02,che,165.0,9500139.0
4,2023-01-03,che,29.0,1270.0
5,2023-01-03,che,105.0,6156.0


# Extract biowaste info for each meal from historical POS data

In [11]:
meal_ids = dim_meal_names['meal_id'].unique()

dim_wastes = pd.DataFrame({
    'meal_id': meal_ids,
    'waste': 0.,
    'embd': [embd for embd in np.eye(len(meal_ids))]
})


dim_wastes.head()

,meal_id,waste,embd
0,9017,0.0,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,7201,0.0,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,9032,0.0,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,9102,0.0,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,7010,0.0,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."


In [12]:
waste = (
    pos
    .merge(dim_wastes, on='meal_id', how='left')
)
waste['embd'] = waste['embd'] * waste['pcs']



waste_byday = (
    waste
    .groupby(['date', 'restaurant'])['embd']
    .sum()
    .reset_index()
    .merge(biowaste_day, on=['date', 'restaurant'], how='inner')
)

waste_byday.head()

,date,restaurant,embd,waste
0,2023-01-02,che,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",17.90
1,2023-01-03,che,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",21.20
2,2023-01-04,che,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",15.25
3,2023-01-05,che,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",21.80
4,2023-01-09,che,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",14.65


In [13]:
X = np.stack(waste_byday['embd'].to_numpy())
y = waste_byday['waste']

model = LinearRegression(positive=True, fit_intercept=False)
model.fit(X, y)


dim_wastes['waste'] = model.coef_.copy()
dim_wastes.drop(columns='embd', inplace=True)

dim_wastes.head()

,meal_id,waste
0,9017,0.000000
1,7201,0.000000
2,9032,0.000000
3,9102,0.000000
4,7010,0.057843


# Test

### Assert high sale meals have positive waste

In [14]:
ids = pos[pos['pcs'] > 300]['meal_id'].unique().astype(int)

tmp = dim_wastes[dim_wastes['meal_id'].isin(ids)].copy()
tmp['valid'] = (tmp['waste'] < 0).astype(int)

assert tmp['valid'].sum() == 0

# Save

In [15]:
path = "data/processed/phase_4/dim_waste.xlsx"
dim_wastes.to_excel(path, index=False)